# Chapter 07: The GitHub API In Depth & GitHub Apps (Reference)

## Learning Objectives

- Walk a paginated list endpoint to completion
- Recognize GitHub's file-count truncation ceiling
- Construct a check-run publish call
- State the structural difference between a PAT and a GitHub App

## Setup

The next cell sets up reproducibility, the `PRA_MODE` fixture/live toggle, and inserts the repo root onto `sys.path` so `pr_automerge` is importable. You should see `PRA_MODE = 'fixture'` printed (unless you've set it to `live`).

In [ ]:
import os
import random
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "pr_automerge").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

# Reproducibility -- always set before any stochastic operation
RANDOM_STATE: int = 42
random.seed(RANDOM_STATE)

# PRA_ environment variables -- fixture mode by default so this notebook runs
# identically for every reader. Set PRA_MODE=live and PRA_REPO=owner/name before
# starting Jupyter to run this against the real sandbox repo instead.
PRA_MODE = os.environ.get("PRA_MODE", "fixture")
PRA_REPO = os.environ.get("PRA_REPO", "")
OUTPUT_DIR = Path(os.environ.get("PRA_OUTPUT_DIR", "output"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if PRA_MODE == "live":
    assert PRA_REPO, "Set PRA_REPO=owner/name to run in live mode"

print(f"PRA_MODE = {PRA_MODE!r}")
print(f"RANDOM_STATE = {RANDOM_STATE}")


## 1. Paginated Read

The next cell walks a PR's changed-files list page by page using `fetch_pr_files_paginated` from `labs/lab_07_api_and_apps.py`. You should see every file's path printed, plus a truncation flag.

In [ ]:
from labs.lab_07_api_and_apps import fetch_pr_files_paginated

files, truncated = fetch_pr_files_paginated(1)
print(f"{len(files)} files, truncated={truncated}")
for f in files:
    print(" ", f["path"])

## 2. Constructing a Check-Run Call

The next cell builds (but in fixture mode does not send) the exact `gh api` argument list Section 7's check-run write uses. You should see the full argument list, matching the pattern Gate 2 and Gate 3's workflow YAML use directly.

In [ ]:
args = [
    "api", "repos/OWNER/REPO/check-runs",
    "-f", "name=lab-07-demo",
    "-f", "head_sha=abc123",
    "-f", "status=completed",
    "-f", "conclusion=success",
    "-f", "output[title]=Lab 07 demo",
    "-f", "output[summary]=Constructed in the Chapter 07 notebook",
]
print(" ".join(args))

## 3. PAT vs GitHub App, As Data

The next cell encodes Chapter 07 Section 12's comparison table as a small Python structure, so the trade-offs are something you can inspect rather than only read in prose.

In [ ]:
identities = {
    "GITHUB_TOKEN": {"setup_effort": "Minimal", "control": "Weak", "security_exposure": "Minimal"},
    "PAT": {"setup_effort": "Low", "control": "Strong", "security_exposure": "High"},
    "GitHub App": {"setup_effort": "High", "control": "Excellent", "security_exposure": "Low"},
}
for name, props in identities.items():
    print(f"{name}: {props}")

## Takeaways & Next Steps

This notebook's takeaways are the numbers you just produced above, not abstract claims -- re-read the printed output from each section before moving on.

In [ ]:
print("Re-run this notebook with PRA_MODE=live to see it against the real sandbox repo.")

---

📖 **Reading companion:** [Chapter 07: The GitHub API In Depth & GitHub Apps](../learning_modules/chapter_07_github_api_and_apps.md)
